# Drone Detection via PCA + Nearest-Neighbour Classifier

End-to-end pipeline:
1. **Imports & Configuration** — shared constants used by every cell
2. **Feature Extraction** — shared `extract_features` function (raw-pixel grayscale)
3. **Dataset Loaders** — YOLO-annotated drone crops + random background patches
4. **PCA Training** — Gram-matrix surrogate trick (avoids 4096×4096 covariance)
5. **Save / Load Model** — persist `mean_vector`, `eigenvectors`, `Z_train_*`
6. **Sliding-Window Detection** — multi-scale, variance-filtered, NN-classified
7. **Batch Test** — run on 100 YOLO images, save annotated JPEGs + bounding-box TXTs

## 1 — Imports & Configuration

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Paths ────────────────────────────────────────────────────────────
YOLO_DIR        = './archive/drone_dataset_yolo/dataset_txt'
BG_DIR          = './archive/Dataset/train/images'
OUTPUT_DIR      = 'data/res_1'
PARAMS_DIR      = 'model_params'

# ── Feature space ────────────────────────────────────────────────────
IMG_SIZE        = (64, 64)          # patch size fed to PCA
FLAT_SIZE       = IMG_SIZE[0] * IMG_SIZE[1]

# ── PCA ──────────────────────────────────────────────────────────────
VARIANCE_THR    = 0.97              # stop adding components after this
DRONE_LIMIT     = 1200              # max drone patches to train on

# ── Sliding-window detector ──────────────────────────────────────────
SCALES          = (1.0, 0.75, 0.5, 0.35, 0.25, 0.15, 0.10)
STRIDE          = 16               # pixels between window positions
STD_THRESHOLD   = 15.0             # skip flat (sky) patches
NN_MARGIN       = 0.25             # d_bg - d_drone must exceed this
NMS_IOU         = 0.5
TOP_K           = 8                # max detections per image
NUM_TEST_IMAGES = 100

## 2 — Feature Extraction

> **Rule:** `extract_features` is the *only* place that converts pixels → feature vector.
> It must be identical between training (PCA) and inference (detector).

In [ ]:
def extract_features(image, target_size=IMG_SIZE):
    """
    Resize to target_size, convert to grayscale, flatten, normalise to [0, 1].
    Accepts both BGR colour images and already-grayscale patches.
    Returns None for empty/None inputs.
    """
    if image is None or image.size == 0:
        return None
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    resized = cv2.resize(gray, target_size, interpolation=cv2.INTER_AREA)
    return resized.flatten() / 255.0

## 3 — Dataset Loaders

In [ ]:
def load_drone_patches(dataset_folder, limit=DRONE_LIMIT):
    """Load annotated drone crops from a YOLO-format folder."""
    patches = []
    for filename in sorted(os.listdir(dataset_folder)):
        if not filename.endswith(('.jpg', '.png')):
            continue
        img_path = os.path.join(dataset_folder, filename)
        txt_path = img_path.rsplit('.', 1)[0] + '.txt'
        if not os.path.exists(txt_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        H, W = img.shape[:2]
        with open(txt_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, cx, cy, bw, bh = map(float, parts)
                x1, y1 = int((cx - bw/2)*W), int((cy - bh/2)*H)
                x2, y2 = int((cx + bw/2)*W), int((cy + bh/2)*H)
                crop = img[max(0, y1):min(H, y2), max(0, x1):min(W, x2)]
                feat = extract_features(crop)
                if feat is not None:
                    patches.append(feat)
        if len(patches) >= limit:
            break
    return np.array(patches[:limit])


def load_background_patches(dataset_folder, n_patches=None, seed=42):
    """Sample random 64×64 patches from background images."""
    patches = []
    rng   = np.random.default_rng(seed)
    files = sorted([f for f in os.listdir(dataset_folder) if f.endswith(('.jpg', '.png'))])
    while len(patches) < (n_patches or DRONE_LIMIT):
        filename = rng.choice(files)
        img = cv2.imread(os.path.join(dataset_folder, filename))
        if img is None:
            continue
        H, W = img.shape[:2]
        if H < 64 or W < 64:
            continue
        x = rng.integers(0, W - 64)
        y = rng.integers(0, H - 64)
        feat = extract_features(img[y:y+64, x:x+64])
        if feat is not None:
            patches.append(feat)
    return np.array(patches[:n_patches])


print('Loading drone patches ...')
X_drone = load_drone_patches(YOLO_DIR)
print(f'  {X_drone.shape[0]} drone patches  shape={X_drone.shape}')

print('Loading background patches ...')
X_bg = load_background_patches(BG_DIR, n_patches=X_drone.shape[0])
print(f'  {X_bg.shape[0]} background patches  shape={X_bg.shape}')

## 4 — PCA (Gram-matrix  trick)

Instead of computing the `D×D` covariance matrix (4096×4096), we form the `N×N`
 Gram matrix `C = X_c X_c^T / N` and run power iteration on it.
Eigenvectors are then back-projected to D-space via `v_D = X_c^T v_N / ‖…‖`.

In [ ]:
def build_pca(X, variance_threshold=VARIANCE_THR, seed=42):
    """
    Returns (mean_vector, eigenvectors) where eigenvectors has shape (D, K).
    Uses the N×N  Gram matrix to keep memory tractable.
    """
    n, d = X.shape
    mean_vec  = X.mean(axis=0)
    X_c       = X - mean_vec
    C_p    = X_c @ X_c.T / n       # N×N Gram matrix
    total_var = np.trace(C_surr)
    cumul_var = 0.0
    rng       = np.random.default_rng(seed)
    evecs_n   = []   # eigenvectors in N-space
    evecs_d   = []   # back-projected eigenvectors in D-space
    evals     = []

    for _ in range(n):
        if total_var > 0 and cumul_var / total_var >= variance_threshold:
            break
        # Random init + deflation (Gram-Schmidt)
        v = rng.standard_normal(n)
        v /= np.linalg.norm(v) + 1e-12
        for u in evecs_n:
            v -= (v @ u) * u
        v /= np.linalg.norm(v) + 1e-12
        # Power iteration
        for _ in range(60):
            v_new = C_p @ v
            for u in evecs_n:
                v_new -= (v_new @ u) * u
            vn = np.linalg.norm(v_new)
            if vn < 1e-12:
                break
            v_new /= vn
            if min(np.linalg.norm(v_new - v), np.linalg.norm(v_new + v)) < 1e-7:
                break
            v = v_new
        eigval = float(v @ C_surr @ v)
        evecs_n.append(v)
        evals.append(max(0.0, eigval))
        cumul_var += eigval
        # Back-project to D-space
        y  = X_c.T @ v
        yn = np.linalg.norm(y)
        if yn > 1e-12:
            evecs_d.append(y / yn)

    print(f'  PCA: kept {len(evecs_d)} components ({100*cumul_var/total_var:.1f}% variance)')
    return mean_vec, np.column_stack(evecs_d), np.array(evals)


print('Training PCA ...')
mean_vec, evecs, evals = build_pca(X_drone)
print(f'  Eigenvector matrix shape: {evecs.shape}')

## 5 — Project Reference Sets & Save Model

In [ ]:
Z_drone = ((X_drone - mean_vec) @ evecs).astype(np.float64)
Z_bg    = ((X_bg    - mean_vec) @ evecs).astype(np.float64)

os.makedirs(PARAMS_DIR, exist_ok=True)
np.save(f'{PARAMS_DIR}/mean_vector.npy',       mean_vec)
np.save(f'{PARAMS_DIR}/eigenvectors.npy',       evecs)
np.save(f'{PARAMS_DIR}/Z_train_drone.npy',      Z_drone)
np.save(f'{PARAMS_DIR}/Z_train_non_drone.npy',  Z_bg)
print('Model saved to', PARAMS_DIR)

cumulative = np.cumsum(evals) / evals.sum()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cumulative, linewidth=1.5)
ax.axhline(VARIANCE_THR, color='r', linestyle='--', label=f'{VARIANCE_THR*100:.0f}% threshold')
ax.set_xlabel('Number of components'); ax.set_ylabel('Cumulative variance')
ax.set_title('PCA Scree Plot'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6 — Load Saved Model

> Run this cell if you restart the kernel and want to skip re-training.

In [ ]:
mean_vec = np.load(f'{PARAMS_DIR}/mean_vector.npy').astype(np.float64)
evecs    = np.load(f'{PARAMS_DIR}/eigenvectors.npy').astype(np.float64)
Z_drone  = np.load(f'{PARAMS_DIR}/Z_train_drone.npy').astype(np.float64)
Z_bg     = np.load(f'{PARAMS_DIR}/Z_train_non_drone.npy').astype(np.float64)
print(f'Loaded: mean={mean_vec.shape}, evecs={evecs.shape}, '
      f'Z_drone={Z_drone.shape}, Z_bg={Z_bg.shape}')

## 7 — Detector Helper Functions

In [ ]:
def _min_dist(A, B, chunk=512):
    """
    Vectorised minimum L2 distance from each row of A (M,K) to any row of B (N,K).
    Uses chunked dot-product to avoid float32 overflow on large reference sets.
    """
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    A_sq  = np.einsum('ij,ij->i', A, A)[:, None]
    B_sq  = np.einsum('ij,ij->i', B, B)[None, :]
    min_d = np.full(len(A), np.inf)
    for start in range(0, len(B), chunk):
        AB = A @ B[start:start+chunk].T
        d2 = np.maximum(A_sq + B_sq[:, start:start+chunk] - 2.0 * AB, 0.0)
        min_d = np.minimum(min_d, d2.min(axis=1))
    return np.sqrt(min_d)


def _iou(b1, b2):
    ax1, ay1, ax2, ay2 = b1[0], b1[1], b1[0]+b1[2], b1[1]+b1[3]
    bx1, by1, bx2, by2 = b2[0], b2[1], b2[0]+b2[2], b2[1]+b2[3]
    ix = max(0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter / union if union > 0 else 0.0


def nms(detections, iou_thresh=NMS_IOU):
    """Greedy NMS. Score = d_drone (lower = better match)."""
    if not detections:
        return []
    detections = sorted(detections, key=lambda d: d[0])
    kept = []
    while detections:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections if _iou(best[1:], d[1:]) < iou_thresh]
    return kept


def detect_single(img_gray, mean_vec, evecs, Z_drone, Z_bg):
    """
    Multi-scale sliding-window detector for a single grayscale image.
    Returns list of (score, x, y, w, h) in original image coordinates.
    """
    ph, pw     = IMG_SIZE
    detections = []

    for s in SCALES:
        sh = int(img_gray.shape[0] * s)
        sw = int(img_gray.shape[1] * s)
        if sh < ph or sw < pw:
            continue
        simg = cv2.resize(img_gray, (sw, sh), interpolation=cv2.INTER_AREA)

        feats, pos = [], []
        for y in range(0, sh - ph, STRIDE):
            for x in range(0, sw - pw, STRIDE):
                raw = simg[y:y+ph, x:x+pw]
                # Variance filter: skip flat sky / plain-wall patches
                if np.std(raw) < STD_THRESHOLD:
                    continue
                feat = extract_features(raw)
                if feat is not None:
                    feats.append(feat)
                    pos.append((x, y))

        if not feats:
            continue

        Z       = (np.array(feats, dtype=np.float64) - mean_vec) @ evecs
        d_drone = _min_dist(Z, Z_drone)
        d_bg    = _min_dist(Z, Z_bg)

        mask = (d_bg - d_drone) > NN_MARGIN
        for idx in np.where(mask)[0]:
            x, y = pos[idx]
            detections.append((
                d_drone[idx],
                int(x / s), int(y / s),
                int(pw / s), int(ph / s),
            ))

    return nms(detections)[:TOP_K]

## 8 — Single-Image Detection Demo

In [ ]:
DEMO_IMAGE = os.path.join(YOLO_DIR, '0001.jpg')

img  = cv2.imread(DEMO_IMAGE)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
kept = detect_single(gray, mean_vec, evecs, Z_drone, Z_bg)

# Draw detections
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(img_rgb)
for score, x, y, w, h in kept:
    rect = mpatches.Rectangle((x, y), w, h, linewidth=2,
                               edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, max(0, y-4), f'{score:.2f}', color='lime',
            fontsize=7, fontweight='bold')
ax.axis('off')
ax.set_title(f'{os.path.basename(DEMO_IMAGE)} — {len(kept)} detection(s)')
plt.tight_layout(); plt.show()
print('Detections (score, x, y, w, h):')
for d in kept:
    print(f'  score={d[0]:.4f}  box=({d[1]}, {d[2]}, {d[3]}, {d[4]})')

## 9 — Batch Test on 100 YOLO Images

Saves annotated images and coordinate files to `data/res_1/`.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_files  = sorted([f for f in os.listdir(YOLO_DIR) if f.endswith(('.jpg', '.png'))])
test_files = all_files[:NUM_TEST_IMAGES]
print(f'Running on {len(test_files)} images ...')

for i, filename in enumerate(test_files):
    img = cv2.imread(os.path.join(YOLO_DIR, filename))
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kept = detect_single(gray, mean_vec, evecs, Z_drone, Z_bg)
    base = os.path.splitext(filename)[0]

    # Annotate and save image
    img_out = img.copy()
    for score, x, y, w, h in kept:
        cv2.rectangle(img_out, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img_out, f'drone {score:.2f}', (x, max(0, y-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    cv2.imwrite(os.path.join(OUTPUT_DIR, f'{base}.jpg'), img_out)

    # Save bounding-box text file: score x y w h
    with open(os.path.join(OUTPUT_DIR, f'{base}.txt'), 'w') as f:
        for score, x, y, w, h in kept:
            f.write(f'{score:.4f} {x} {y} {w} {h}\n')

    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/{len(test_files)}')

print(f'Done — results saved to {OUTPUT_DIR}/')